# Unidad 3 – Sesión 1
## Limpieza y tratamiento inicial de datos

**Curso:** Minería de Datos  
**Fase CRISP-DM:** Preparación de los Datos  

### Propósito
Al finalizar la sesión, el estudiante será capaz de:
- identificar problemas que requieren preparación;
- tratar nulos, duplicados, inconsistencias, valores inválidos y outliers;
- justificar cada decisión;
- validar el resultado antes y después de transformar los datos.

> **Idea clave:** En la Unidad 2 detectamos los problemas. En la Unidad 3 comenzamos a resolverlos.

# 1. Dataset de ejemplo

Usaremos un dataset pequeño con problemas intencionales de calidad.

In [45]:
import pandas as pd
import numpy as np

data = {
    "cliente_id": [1, 2, 3, 3, 4, 5, 6, 7],
    "edad": [25, 32, None, 40, 250, 28, -5, 36],
    "ciudad": ["Lima", "LIMA", "lima", "Cusco", "Cusco", None, "LIMA ", "Arequipa"],
    "ingreso": [1500, 2300, None, 1800, 50000, 1700, 1600, 1750],
    "estado": ["Si", "Sí", "SI", "No", "NO", "yes", "NO", "SI"]
}

df = pd.DataFrame(data)
df

,cliente_id,edad,ciudad,ingreso,estado
0,1,25.0,Lima,1500.0,Si
1,2,32.0,LIMA,2300.0,Sí
2,3,NaN,lima,NaN,SI
3,3,40.0,Cusco,1800.0,No
4,4,250.0,Cusco,50000.0,NO
5,5,28.0,None,1700.0,yes
6,6,-5.0,LIMA,1600.0,NO
7,7,36.0,Arequipa,1750.0,SI


### Iniciamos....
1. ¿Qué problemas detectas?
2. ¿Qué variables presentan nulos?
3. ¿Hay categorías inconsistentes?
4. ¿Hay valores inválidos?
5. ¿Hay posibles outliers?
6. ¿Existen posibles duplicados?

# 2. Detectar vs preparar

En la Unidad 2 diagnosticábamos:

```python
df.isnull().sum()
df.duplicated().sum()
df["ciudad"].value_counts()
df.describe()
```

En la Unidad 3 tomamos decisiones y transformamos:

```python
df["ciudad"] = df["ciudad"].str.strip().str.upper()
```

La secuencia será:

**Problema → Evidencia → Decisión → Transformación → Validación**

In [2]:
plan_ejemplo = pd.DataFrame({
    "Problema": [
        "Categoría inconsistente",
        "Edad nula",
        "Posible duplicado",
        "Edad inválida"
    ],
    "Evidencia": [
        "Lima / LIMA / lima / LIMA ",
        "Existe al menos un valor nulo",
        "cliente_id = 3 aparece más de una vez",
        "Edad = 250 y Edad = -5"
    ],
    "Decision": [
        "Estandarizar",
        "Imputar",
        "Revisar llave lógica",
        "Convertir a nulo"
    ],
    "Transformacion": [
        "strip() + upper()",
        "Mediana",
        "duplicated(subset=...)",
        "Regla de rango"
    ],
    "Validacion": [
        "value_counts()",
        "isnull().sum()",
        "Volver a contar duplicados",
        "describe()"
    ]
})
plan_ejemplo

,Problema,Evidencia,Decision,Transformacion,Validacion
0,Categoría inconsistente,Lima / LIMA / lima / LIMA,Estandarizar,strip() + upper(),value_counts()
1,Edad nula,Existe al menos un valor nulo,Imputar,Mediana,isnull().sum()
2,Posible duplicado,cliente_id = 3 aparece más de una vez,Revisar llave lógica,duplicated(subset=...),Volver a contar duplicados
3,Edad inválida,Edad = 250 y Edad = -5,Convertir a nulo,Regla de rango,describe()


In [5]:
df.duplicated().sum()

np.int64(0)

# 3. Tratamiento de valores nulos

In [3]:
df.isnull().sum()

cliente_id    0
edad          1
ciudad        1
ingreso       1
estado        0
dtype: int64

In [4]:
(df.isnull().mean() * 100).round(2)

cliente_id     0.0
edad          12.5
ciudad        12.5
ingreso       12.5
estado         0.0
dtype: float64

## Eliminar filas
Puede utilizarse cuando los nulos son pocos y la pérdida de registros no afecta de forma importante el análisis.

In [6]:
df_sin_nulos = df.dropna()
df_sin_nulos

,cliente_id,edad,ciudad,ingreso,estado
0,1,25.0,Lima,1500.0,Si
1,2,32.0,LIMA,2300.0,Sí
3,3,40.0,Cusco,1800.0,No
4,4,250.0,Cusco,50000.0,NO
6,6,-5.0,LIMA,1600.0,NO
7,7,36.0,Arequipa,1750.0,SI


## Imputación numérica
La mediana suele ser más robusta cuando existen valores extremos.

In [7]:
df_imputado = df.copy()
df_imputado["ingreso"] = df_imputado["ingreso"].fillna(df_imputado["ingreso"].median())
df_imputado["ingreso"].isnull().sum()

np.int64(0)

## Imputación categórica
Se puede usar la moda o una categoría explícita como `"DESCONOCIDO"`.

In [8]:
df_categorico = df.copy()
df_categorico["ciudad"] = df_categorico["ciudad"].fillna("DESCONOCIDO")
df_categorico["ciudad"].value_counts(dropna=False)

ciudad
Cusco          2
Lima           1
LIMA           1
lima           1
DESCONOCIDO    1
LIMA           1
Arequipa       1
Name: count, dtype: int64

¿Qué ventaja tiene usar `"DESCONOCIDO"` en vez de reemplazar automáticamente por la categoría más frecuente?

# 4. Duplicados

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df[df.duplicated(subset=["cliente_id"], keep=False)]

,cliente_id,edad,ciudad,ingreso,estado
2,3,NaN,lima,NaN,SI
3,3,40.0,Cusco,1800.0,No


> Un identificador repetido no implica automáticamente un error. Primero debe verificarse si la granularidad exige una sola fila por cliente.

# 5. Consistencia de variables categóricas

In [11]:
df["ciudad"].value_counts(dropna=False)

ciudad
Cusco       2
Lima        1
LIMA        1
lima        1
None        1
LIMA        1
Arequipa    1
Name: count, dtype: int64

In [12]:
df_limpio = df.copy()

df_limpio["ciudad"] = (
    df_limpio["ciudad"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_limpio["ciudad"].value_counts(dropna=False)

ciudad
LIMA        4
CUSCO       2
<NA>        1
AREQUIPA    1
Name: count, dtype: Int64

In [13]:
df_limpio["estado"] = (
    df_limpio["estado"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({
        "SÍ": "SI",
        "YES": "SI"
    })
)

df_limpio["estado"].value_counts(dropna=False)

estado
SI    5
NO    3
Name: count, dtype: Int64

# 6. Valores inválidos

In [14]:
df["edad"].describe()

count      7.000000
mean      58.000000
std       85.932144
min       -5.000000
25%       26.500000
50%       32.000000
75%       38.000000
max      250.000000
Name: edad, dtype: float64

In [15]:
df_validacion = df.copy()

df_validacion.loc[
    (df_validacion["edad"] < 0) |
    (df_validacion["edad"] > 100),
    "edad"
] = np.nan

df_validacion["edad"].describe()

count     5.000000
mean     32.200000
std       6.016644
min      25.000000
25%      28.000000
50%      32.000000
75%      36.000000
max      40.000000
Name: edad, dtype: float64

In [16]:
df_validacion["edad"] = df_validacion["edad"].fillna(
    df_validacion["edad"].median()
)

df_validacion["edad"].isnull().sum()

np.int64(0)

# 7. Outliers

> **Un outlier no significa automáticamente que exista un error.**

In [17]:
Q1 = df["ingreso"].quantile(0.25)
Q3 = df["ingreso"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)

Q1: 1650.0
Q3: 2050.0
IQR: 400.0
Límite inferior: 1050.0
Límite superior: 2650.0


In [18]:
outliers_ingreso = df[
    (df["ingreso"] < limite_inferior) |
    (df["ingreso"] > limite_superior)
]

outliers_ingreso

,cliente_id,edad,ciudad,ingreso,estado
4,4,250.0,Cusco,50000.0,NO


### Posibles decisiones
- Conservar
- Corregir
- Transformar
- Excluir
- Analizar por separado

La decisión siempre debe estar justificada.

# 8. Validación antes y después

In [19]:
df_validado = df.copy()

print("ANTES")
print(df_validado["ciudad"].value_counts(dropna=False))

df_validado["ciudad"] = (
    df_validado["ciudad"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("\nDESPUÉS")
print(df_validado["ciudad"].value_counts(dropna=False))

ANTES
ciudad
Cusco       2
Lima        1
LIMA        1
lima        1
None        1
LIMA        1
Arequipa    1
Name: count, dtype: int64

DESPUÉS
ciudad
LIMA        4
CUSCO       2
<NA>        1
AREQUIPA    1
Name: count, dtype: Int64


In [20]:
print("Nulos en edad ANTES:", df["edad"].isnull().sum())

df_validado["edad"] = df["edad"].copy()

df_validado.loc[
    (df_validado["edad"] < 0) |
    (df_validado["edad"] > 100),
    "edad"
] = np.nan

df_validado["edad"] = df_validado["edad"].fillna(
    df_validado["edad"].median()
)

print("Nulos en edad DESPUÉS:", df_validado["edad"].isnull().sum())

Nulos en edad ANTES: 1
Nulos en edad DESPUÉS: 0


# 9. Actividad práctica

Carga tu propio dataset.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [22]:
# Para CSV:
df_proyecto = pd.read_csv("LI-Small_Trans.csv")

# Para Excel:
# df_proyecto = pd.read_excel("archivo.xlsx")

- 2 variables con nulos;


In [28]:
nulos = df_proyecto.isnull().sum()
nulos[nulos > 0]

Series([], dtype: int64)

In [29]:
df_proyecto[df_proyecto.duplicated(keep=False)]

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
676694,2022/09/01 05:03,21260,8085A09D1,72146,81AB967E1,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
676696,2022/09/01 05:03,21260,8085A09D1,72146,81AB967E1,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
790828,2022/09/01 07:44,231616,814E45511,171546,81B324AC1,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
790830,2022/09/01 07:44,231616,814E45511,171546,81B324AC1,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
2027614,2022/09/02 11:09,28335,804B468F1,171669,81BBCDAB1,0.000007,Bitcoin,0.000007,Bitcoin,Bitcoin,0
2027616,2022/09/02 11:09,28335,804B468F1,171669,81BBCDAB1,0.000007,Bitcoin,0.000007,Bitcoin,Bitcoin,0
3482551,2022/09/05 12:44,22661,80D016C81,269294,81A245E81,0.000005,Bitcoin,0.000005,Bitcoin,Bitcoin,0
3482553,2022/09/05 12:44,22661,80D016C81,269294,81A245E81,0.000005,Bitcoin,0.000005,Bitcoin,Bitcoin,0
4180129,2022/09/06 14:16,19970,804E44F11,169489,8197DC971,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
4180131,2022/09/06 14:16,19970,804E44F11,169489,8197DC971,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0


In [30]:
# Revisar categorías y detectar diferencias de escritura o espacios
columnas = ["Receiving Currency", "Payment Currency", "Payment Format"]

for columna in columnas:
    print(f"\n{columna}:")
    print(df_proyecto[columna].value_counts(dropna=False))


Receiving Currency:
Receiving Currency
US Dollar            2537242
Euro                 1596407
Yuan                  474978
Rupee                 344237
Bitcoin               313196
Saudi Riyal           261882
Australian Dollar     213905
Yen                   211631
Brazil Real           202717
Canadian Dollar       177966
Shekel                177298
Swiss Franc           140076
UK Pound               98000
Ruble                  89140
Mexican Peso           85374
Name: count, dtype: int64

Payment Currency:
Payment Currency
US Dollar            2553887
Euro                 1595859
Yuan                  483603
Rupee                 340641
Bitcoin               309240
Saudi Riyal           257948
Australian Dollar     211155
Yen                   210125
Brazil Real           199840
Canadian Dollar       176069
Shekel                174530
Swiss Franc           138251
UK Pound               99668
Ruble                  88492
Mexican Peso           84741
Name: count, dtype: int64

P

In [32]:
# Resumen estadístico de los montos
df_proyecto[["Amount Received", "Amount Paid"]].describe()

,Amount Received,Amount Paid
count,6.924049e+06,6.924049e+06
mean,6.324067e+06,4.676036e+06
std,2.105371e+09,1.544099e+09
min,1.000000e-06,1.000000e-06
25%,1.742100e+02,1.753800e+02
50%,1.397620e+03,1.399440e+03
75%,1.229633e+04,1.222687e+04
max,3.644854e+12,3.644854e+12


In [33]:
df_proyecto.groupby("Receiving Currency")["Amount Received"].describe()

,count,mean,std,min,25%,50%,75%,max
Receiving Currency,,,,,,,,
Australian Dollar,213905.0,4.265410e+05,1.392287e+07,0.010000,216.650000,1221.790000,7145.080000,2.650864e+09
Bitcoin,313196.0,2.846249e+01,1.558807e+03,0.000001,0.011749,0.069771,0.419921,6.160250e+05
Brazil Real,202717.0,1.908360e+06,5.532194e+07,0.010000,940.370000,5389.770000,30937.820000,1.410079e+10
Canadian Dollar,177966.0,3.930941e+05,1.465926e+07,0.010000,212.060000,1192.115000,6974.700000,2.709593e+09
Euro,1596407.0,2.650207e+05,2.149714e+07,0.010000,134.670000,752.920000,4271.600000,2.348374e+10
Mexican Peso,85374.0,1.219701e+07,1.021383e+09,0.010000,3329.590000,19233.780000,114746.130000,1.503626e+11
Ruble,89140.0,4.075813e+07,1.057353e+09,0.010000,12237.720000,72567.325000,418018.480000,8.698050e+10
Rupee,344237.0,3.628159e+07,2.305267e+09,0.010000,11508.170000,66719.920000,431843.580000,8.623400e+11
Saudi Riyal,261882.0,1.138538e+06,4.620901e+07,0.010000,613.630000,3514.290000,20619.100000,1.513349e+10


In [34]:
df_proyecto[
    df_proyecto["Is Laundering"].notna() &
    ~df_proyecto["Is Laundering"].isin([0, 1])
]

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering


In [36]:
montos = df_proyecto.loc[
    df_proyecto["Receiving Currency"] == "US Dollar",
    "Amount Received"
]

Q1 = montos.quantile(0.25)
Q3 = montos.quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df_proyecto.loc[
    montos[
        (montos < limite_inferior) |
        (montos > limite_superior)
    ].index
]

outliers.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
6,2022/09/01 00:17,11,800059120,1217,8006AD4E0,60562.00,US Dollar,60562.00,US Dollar,ACH,0
13,2022/09/01 00:08,394,80093BDB0,21414,80092A6C0,64335.00,US Dollar,64335.00,US Dollar,ACH,0
16,2022/09/01 00:14,394,80093BDB0,394,80093BDB0,14903.78,US Dollar,14903.78,US Dollar,Reinvestment,0


Identifica como mínimo:
- 2 variables con nulos;
- 1 posible duplicado;
- 1 problema de consistencia;
- 1 valor inválido;
- 1 posible outlier.

In [41]:
plan_preparacion = pd.DataFrame({
    "Variable": [
        "Todas las variables",
        "Transacción completa",
        "Receiving Currency / Payment Currency / Payment Format",
        "Amount Received / Amount Paid",
        "Is Laundering",
        "Amount Received"
    ],
"Problema": [
    "No hay",
    "Posibles duplicados",
    "No hay",
    "No hay",
    "No hay",
    "Posible outlier"
],
    "Evidencia": [
        "Todas las variables tienen cero nulos",
        "Los registros 676694 y 676696 coinciden en todas sus columnas",
        "Las categorías revisadas no presentan diferencias de escritura",
        "No hay montos negativos ni ceros",
        "Solo contiene 0 y 1",
        "Registro 0: 3 195 403 USD supera el límite IQR de 12 377.16 USD"
    ],
    "Decision": [
        "Conservar",
        "Revisar antes de eliminar",
        "Conservar",
        "Conservar",
        "Conservar",
        "Conservar y revisar"
    ],
    "Transformacion": [
        "Ninguna",
        "Pendiente: marcar posibles duplicados",
        "Ninguna",
        "Ninguna",
        "Ninguna",
        "Pendiente: marcar el posible outlier"
    ],
    "Justificacion": [
        "No hay nulos que tratar",
        "Podrían ser operaciones diferentes realizadas en el mismo minuto",
        "No se detectaron inconsistencias que corregir",
        "No se detectaron incumplimientos de las reglas revisadas",
        "Los valores pertenecen al dominio permitido",
        "Ser un valor extremo no demuestra que sea un error"
    ]
})

plan_preparacion

,Variable,Problema,Evidencia,Decision,Transformacion,Justificacion
0,Todas las variables,No hay,Todas las variables tienen cero nulos,Conservar,Ninguna,No hay nulos que tratar
1,Transacción completa,Posibles duplicados,Los registros 676694 y 676696 coinciden en tod...,Revisar antes de eliminar,Pendiente: marcar posibles duplicados,Podrían ser operaciones diferentes realizadas ...
2,Receiving Currency / Payment Currency / Paymen...,No hay,Las categorías revisadas no presentan diferenc...,Conservar,Ninguna,No se detectaron inconsistencias que corregir
3,Amount Received / Amount Paid,No hay,No hay montos negativos ni ceros,Conservar,Ninguna,No se detectaron incumplimientos de las reglas...
4,Is Laundering,No hay,Solo contiene 0 y 1,Conservar,Ninguna,Los valores pertenecen al dominio permitido
5,Amount Received,Posible outlier,Registro 0: 3 195 403 USD supera el límite IQR...,Conservar y revisar,Pendiente: marcar el posible outlier,Ser un valor extremo no demuestra que sea un e...


### Decisiones de preparación

1. **Valores nulos**
   - **Problema:** No se encontraron valores nulos.
   - **Evidencia:** El conteo de nulos fue cero en todas las variables.
   - **Decisión:** Conservar los datos sin imputar.
   - **Transformación aplicada:** Ninguna.
   - **Justificación:** No existen valores faltantes que requieran tratamiento.

2. **Posibles duplicados**
   - **Problema:** Se encontraron filas idénticas.
   - **Evidencia:** Los registros 676694 y 676696 coinciden en todas sus columnas.
   - **Decisión:** Conservarlos para revisión antes de eliminarlos.
   - **Transformación aplicada:** Ninguna; se identificaron mediante `duplicated(keep=False)`.
   - **Justificación:** Podrían ser operaciones diferentes realizadas en el mismo minuto, ya que no existe un identificador único de transacción.

3. **Consistencia categórica**
   - **Problema:** No se detectaron inconsistencias en las categorías revisadas.
   - **Evidencia:** Las monedas y los formatos de pago no mostraron diferencias de escritura.
   - **Decisión:** Conservar las categorías.
   - **Transformación aplicada:** Ninguna.
   - **Justificación:** No se encontraron inconsistencias que requieran corrección.

4. **Valores inválidos**
   - **Problema:** No se detectaron valores inválidos con las reglas revisadas.
   - **Evidencia:** Los montos son positivos y `Is Laundering` solo contiene 0 y 1.
   - **Decisión:** Conservar los valores.
   - **Transformación aplicada:** Ninguna.
   - **Justificación:** Los valores cumplen las reglas de validación revisadas.

5. **Posible outlier**
   - **Problema:** Se identificó un monto recibido estadísticamente extremo.
   - **Evidencia:** El registro 0 contiene 3 195 403 USD y supera el límite superior IQR de aproximadamente 12 377.16 USD.
   - **Decisión:** Conservarlo para revisión.
   - **Transformación aplicada:** Ninguna; se identificó mediante la regla IQR para transacciones en dólares.
   - **Justificación:** Un monto extremo no implica necesariamente un error y puede ser relevante para detectar operaciones inusuales.

# 10. Matriz final de preparación

In [44]:
matriz_final = pd.DataFrame({
    "Variable": [
        "Todas las variables",
        "Transacción completa",
        "Receiving Currency / Payment Currency / Payment Format",
        "Amount Received / Amount Paid / Is Laundering",
        "Amount Received"
    ],
    "Problema": [
        "No hay nulos",
        "Posibles duplicados",
        "No se detectaron inconsistencias",
        "No se detectaron valores inválidos con las reglas revisadas",
        "Posible outlier"
    ],
    "Evidencia": [
        "Cero valores nulos en todas las columnas",
        "Los registros 676694 y 676696 coinciden en todas sus columnas",
        "No se observaron diferencias de escritura en las categorías",
        "Montos positivos y etiquetas con valores 0 y 1",
        "Registro 0: 3 195 403 USD supera el límite IQR de 12 377.16 USD"
    ],
    "Decision": [
        "Conservar sin imputar",
        "Conservar para revisión",
        "Conservar las categorías",
        "Conservar los valores",
        "Conservar para revisión"
    ],
    "Transformacion": [
        "Ninguna",
        "Ninguna; detección con duplicated",
        "Ninguna",
        "Ninguna",
        "Ninguna; detección mediante IQR para US Dollar"
    ],
    "Resultado": [
        "Datos conservados sin imputación",
        "Posibles duplicados identificados y conservados",
        "Categorías conservadas",
        "Valores conservados al cumplir las reglas revisadas",
        "Posible outlier identificado y conservado"
    ]
})

matriz_final

,Variable,Problema,Evidencia,Decision,Transformacion,Resultado
0,Todas las variables,No hay nulos,Cero valores nulos en todas las columnas,Conservar sin imputar,Ninguna,Datos conservados sin imputación
1,Transacción completa,Posibles duplicados,Los registros 676694 y 676696 coinciden en tod...,Conservar para revisión,Ninguna; detección con duplicated,Posibles duplicados identificados y conservados
2,Receiving Currency / Payment Currency / Paymen...,No se detectaron inconsistencias,No se observaron diferencias de escritura en l...,Conservar las categorías,Ninguna,Categorías conservadas
3,Amount Received / Amount Paid / Is Laundering,No se detectaron valores inválidos con las reg...,Montos positivos y etiquetas con valores 0 y 1,Conservar los valores,Ninguna,Valores conservados al cumplir las reglas revi...
4,Amount Received,Posible outlier,Registro 0: 3 195 403 USD supera el límite IQR...,Conservar para revisión,Ninguna; detección mediante IQR para US Dollar,Posible outlier identificado y conservado


# 11. Cierre

1. **¿Eliminar nulos siempre mejora la calidad?**  
   No. Puede provocar la pérdida de información importante e introducir sesgos. La decisión depende de la cantidad de nulos y de su causa.

2. **¿Cuándo usarías media y cuándo mediana?**  
   Usaría la media cuando la distribución sea aproximadamente simétrica y no tenga valores extremos importantes. Usaría la mediana cuando haya asimetría o valores extremos.

3. **¿Un outlier debe eliminarse automáticamente?**  
   No. Puede representar un caso real y relevante, como una transacción inusual. Primero debe investigarse.

4. **¿Dos filas iguales siempre son duplicados?**  
   Son duplicados en sus valores, pero no necesariamente representan la misma operación. En este dataset podrían existir transacciones distintas realizadas en el mismo minuto.

5. **¿Qué riesgo existe al imputar demasiados valores?**  
   Puede reducir artificialmente la variabilidad, introducir sesgos y alterar las relaciones entre variables. En transacciones, imputar montos también podría inventar información financiera.

6. **¿Por qué debemos validar después de limpiar?**  
   Para comprobar que los cambios resolvieron los problemas detectados sin introducir errores ni perder información relevante.

7. **¿Qué diferencia existe entre valor inválido y outlier?**  
   Un valor inválido incumple una regla del dominio o del formato. Un outlier es estadísticamente atípico y puede ser válido o inválido; debe revisarse antes de decidir su tratamiento.

8. **¿Qué ocurriría si estandarizamos incorrectamente una categoría?**  
   Podríamos unir categorías con significados diferentes, perder información y distorsionar los resultados del análisis.

# 12. Conclusión

> **En Comprensión de los Datos identificamos los problemas. En Preparación de los Datos tomamos decisiones para transformar el dataset de manera justificada, reproducible y verificable.**

### Producto esperado
- diagnóstico;
- evidencia;
- decisión;
- transformación;
- validación antes/después;
- matriz final de preparación.